# CardioAgent-Ops — QLoRA Qwen2.5

Notebook portable : **Google Colab** (T4/L4) ou **machine locale** (Windows/Linux).

> ⚠️ Projet académique. Le modèle n'est pas un dispositif médical et ne doit pas être utilisé
> pour un diagnostic.

1. Exécuter la cellule **« Racine du dépôt »** : elle détecte le dépôt, s'y place et le clone
   dans `/content` si nécessaire.
2. Exécuter les cellules dans l'ordre.
3. Télécharger `llm_finetuning/outputs/`, `llm_finetuning/export/` et `llm_finetuning/mlruns.db`.

> Le notebook choisit automatiquement la configuration : **7B** si la VRAM ≥ 12 Go,
> **1.5B local** sinon (GPU 4 Go).

In [ ]:
# Racine du dépôt : fonctionne sur Colab comme en local, quel que soit le dossier courant.
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "llm_finetuning" / "src" / "train.py").exists():
            return candidate
    colab_root = Path("/content/cadrioagent-ops")
    if (colab_root / "llm_finetuning" / "src" / "train.py").exists():
        return colab_root
    subprocess.run(
        ["git", "clone", "https://github.com/Ahmed-jarrayaa/cadrioagent-ops.git", str(colab_root)],
        check=True,
    )
    return colab_root


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
# Matériel et choix de la configuration.
import os
import torch

vram_gb = 0.0
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024**3
    print(f"GPU : {props.name} — {vram_gb:.1f} Go de VRAM")
else:
    print("CUDA indisponible : installer un build CUDA de torch (voir README section 5.4).")

if vram_gb >= 12:
    CONFIG = "llm_finetuning/configs/qlora_qwen25_7b.yaml"
    ADAPTER = "llm_finetuning/outputs/qwen25-7b-cardio/adapter"
else:
    CONFIG = "llm_finetuning/configs/qlora_qwen25_1_5b_local.yaml"
    ADAPTER = "llm_finetuning/outputs/qwen25-1.5b-local/adapter"

print("CONFIG  =", CONFIG)
print("ADAPTER =", ADAPTER)

In [ ]:
# Dépendances : stack Windows locale (sans Unsloth) ou stack Colab/Linux.
import os

requirements = (
    "llm_finetuning/requirements-llm-local.txt"
    if os.name == "nt"
    else "llm_finetuning/requirements-llm.txt"
)
print("Installation :", requirements)
!python -m pip install -U pip -q
!python -m pip install -q -r {requirements}

In [ ]:
# Provenance des sources, construction du dataset et tests.
!python llm_finetuning/data/verify_sources.py --refresh
!python llm_finetuning/data/build_dataset.py
!python llm_finetuning/data/build_dataset.py --check
!python -m pytest llm_finetuning/tests -q

In [ ]:
# Vérification à sec : configuration + dataset, sans télécharger le modèle.
!python -m llm_finetuning.src.train --config {CONFIG} --dry-run

In [ ]:
# Entraînement. Sur 4 Go, commencer par un smoke test : ajouter --max-steps 20.
!python -m llm_finetuning.src.train --config {CONFIG}

In [ ]:
# Évaluation indépendante sur le jeu de test (fine-tuné vs base).
!python -m llm_finetuning.src.evaluate --config {CONFIG} --adapter {ADAPTER} --compare-base

In [ ]:
# Export GGUF : nécessite llama.cpp + cmake (Colab/Linux). Ignoré sur Windows.
import os

if os.name == "nt":
    print("Export GGUF ignoré sur Windows : utiliser Colab/WSL (voir README section 8).")
else:
    !git clone https://github.com/ggerganov/llama.cpp llm_finetuning/export/llama.cpp
    !cmake -S llm_finetuning/export/llama.cpp -B llm_finetuning/export/llama.cpp/build
    !cmake --build llm_finetuning/export/llama.cpp/build --config Release -j
    !python -m llm_finetuning.src.export_gguf --config {CONFIG} --adapter {ADAPTER} --output-dir llm_finetuning/export --quantization q4_k_m

## Récupération des artefacts

- Le GGUF et le `Modelfile` sont dans `llm_finetuning/export/`.
- L'adaptateur est dans `llm_finetuning/outputs/<run>/adapter/`.
- La base MLflow locale est `llm_finetuning/mlruns.db` :
  `mlflow ui --backend-store-uri sqlite:///llm_finetuning/mlruns.db`.

Sur la machine locale :

```powershell
ollama create cardio-llm -f llm_finetuning\export\Modelfile
ollama run cardio-llm
```

> L'intégration Docker (`llm-server`) se fera après validation du fine-tuning.